# 🔬 GIADA IE-P6 — copertura × ricorrenza e operatore locale
12 refit appaiati + 6 operatori locali appresi, tre parent congelati, formule/LUT e 12 deployment tabulati. GPU e Internet; nessun input esterno.
Stesso dt0,1ms e stessi blocchi fisici; quality e velocità separate. Fresh dopo freeze; Task35 diagnostica già aperta. Nessuna promozione automatica.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
WORK=Path('/kaggle/working/giada_ie_p6');WORK.mkdir(parents=True,exist_ok=True)
REPO=WORK/'giada';BRANCH='codex/surrogate-validity-audit'
if not REPO.exists():subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/Zagred47/giada.git',str(REPO)],check=True)
REF=os.environ.get('GIADA_IE_P6_REVISION',BRANCH)
subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin',REF],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
print({'revision':subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip(),'protocol':'IE-P6'})


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts/giada_ie_p6_residual_repair')
assert not OUTPUT.exists(),f'Output già presente: {OUTPUT}'
LOG=WORK/'process.log'
with subprocess.Popen([sys.executable,'-u',str(REPO/'scripts/run_ie_p6_residual_repair.py'),str(OUTPUT)],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as worker,LOG.open('w') as log:
    for line in worker.stdout:
        log.write(line);log.flush()
        if line.startswith('[GIADA IE-P6]'):print(line.rstrip(),flush=True)
    exit_code=worker.wait()
OUTPUT.mkdir(parents=True,exist_ok=True);shutil.copy2(LOG,OUTPUT/'process.log')
path=next((OUTPUT/n for n in ('final_report.json','failure_report.json') if (OUTPUT/n).is_file()),None)
REPORT=json.loads(path.read_text()) if path else {'valid':False,'error':'worker terminated without report','exit':exit_code}
if path is None:(OUTPUT/'failure_report.json').write_text(json.dumps(REPORT,indent=2))
print({k:REPORT.get(k) for k in ('valid','error','device','candidate_promoted')})
print({'timing_cells':len(REPORT.get('rows',[])),'graph_failures':len(REPORT.get('graph_failures',[]))})
if exit_code:print('\n'.join(LOG.read_text(errors='replace').splitlines()[-18:]))
ARCHIVE=shutil.make_archive('/kaggle/working/giada_ie_p6_residual_repair','zip',root_dir=OUTPUT)


In [ ]:
import base64
from IPython.display import Javascript, display
payload=base64.b64encode(Path(ARCHIVE).read_bytes()).decode()
display(Javascript('const s=atob('+json.dumps(payload)+');const b=Uint8Array.from(s,c=>c.charCodeAt(0));const u=URL.createObjectURL(new Blob([b],{type:"application/zip"}));const a=document.createElement("a");a.href=u;a.download="giada_ie_p6_residual_repair.zip";document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(u),10000);'))
